This file creates 2 versions for the ML model training data. 

- ***1st Attempt***: joins the business entries just with the neighborhood data
- ***2nd Attempt***: joins the business entries both with the neighborhood and the municipal community data

## ***Initials***

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline

## ***Load the Data***

In [2]:
business_df = pd.read_csv('C:\\Users\\Giorgos\\Desktop\\HMMY\\10ο Εξάμηνο\\Διπλωματική\\4. Joining Datasets\\3. Joining - Business - Neighborhood Data\\Extracted CSV Files\\business_data.csv')
print(business_df.shape)
print(business_df.columns)

(3882, 16)
Index(['Name', 'Category', 'Latitude', 'Longitude', 'Address', 'Country',
       'City', 'Postal Code', 'Rating', 'Reviews', 'Source', 'Description',
       'NACE Code', 'NACE Description (EN)', 'Municipal_Community',
       'Neighborhood'],
      dtype='object')


In [3]:
neighborhood_df = pd.read_csv('C:\\Users\\Giorgos\\Desktop\\HMMY\\10ο Εξάμηνο\\Διπλωματική\\3. Base Datasets\\3. Data -  Smaller Spatial Units\\1. Neighborhoods\\Extracted CSV Files\\neighborhoods_enriched.csv')
print(neighborhood_df.shape)
print(neighborhood_df.columns)

(48, 12)
Index(['Municipal Community', 'Neighborhood_Greek', 'Neighborhood',
       'Centroid_x', 'Centroid_y', 'Neighborhood_Area_km2', 'Geometry',
       'distance_to_volos_center_km', 'distance_to_volos_port_km',
       'dist_to_main_road_km', 'dist_to_bus_stop_km', 'dist_to_university_km'],
      dtype='object')


In [4]:
municipal_community_df = pd.read_csv('C:\\Users\\Giorgos\\Desktop\\HMMY\\10ο Εξάμηνο\\Διπλωματική\\3. Base Datasets\\2. Data - Municipal Communities\\4. Exploratory Data Analysis\\Extracted CSV Files\\ELSTAT-demographic-economic.csv', index_col=0)
print(municipal_community_df.shape)
print(municipal_community_df.columns)

(76, 22)
Index(['ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ', 'ΔΗΜΟΣ', 'ΔΗΜΟΤΙΚΗ ΕΝΟΤΗΤΑ',
       'ΔΗΜΟΤΙΚΗ ΚΟΙΝΟΤΗΤΑ ΕΛΛ', 'ΔΗΜΟΤΙΚΗ ΚΟΙΝΟΤΗΤΑ', 'Population',
       'Άγαμοι_pct', 'Έγγαμοι_pct', 'Χήροι_pct', 'Διαζευγμένοι_pct',
       'age_0_14_pct', 'age_15_64_pct', 'age_65_plus_pct', 'low_education_pct',
       'medium_education_pct', 'high_education_pct', 'unemployment_rate',
       'labor_force_participation_rate', 'primary_sector_pct',
       'secondary_sector_pct', 'tertiary_sector_pct', 'Area_km2'],
      dtype='object')


In [3]:
nace_all_df = pd.read_csv("C:\\Users\\Giorgos\\Desktop\\HMMY\\10ο Εξάμηνο\\Διπλωματική\\2. General Data - Directories\\3. NACE Data\\NACE_Class_Codes.csv")  
print(nace_all_df.shape)
print(nace_all_df.columns)

(608, 2)
Index(['class code', 'class description'], dtype='object')


## ***Inspect the NACEs***

In [6]:
# Compare NACE codes in business data with reference file
business_nace_codes = set(business_df['NACE Code'].unique())
ref_nace_codes = set(nace_all_df['class code'].unique())

# Get unique Neighborhoods from demographics data
unique_nace_codes = business_df['NACE Code'].unique()
unique_neighborhoods = neighborhood_df['Neighborhood'].unique()

print(f"Unique NACE codes in reference file: {len(ref_nace_codes)}")
print(f"Unique NACE codes in business data: {len(business_nace_codes)}")
print("unique Neighborhoods:", len(unique_neighborhoods))
print("Expected number of unique NACE codes:", len(unique_nace_codes)*len(unique_neighborhoods))

Unique NACE codes in reference file: 608
Unique NACE codes in business data: 306
unique Neighborhoods: 48
Expected number of unique NACE codes: 14688


## ***Add the (NACE, Neighborhood) Pairs***

In [7]:
# Create a DataFrame with all possible combinations (Cartesian product)
combinations_df = pd.MultiIndex.from_product(
    [unique_nace_codes, unique_neighborhoods],
    names=['NACE Code', 'Neighborhood']
).to_frame(index=False)

In [8]:
print(combinations_df.shape)
combinations_df.head()

(14688, 2)


,NACE Code,Neighborhood
0,47.76,Dimini
1,47.76,Xrisi Akti Panagias
2,47.76,Sesklo
3,47.76,Agioi Anargiroi
4,47.76,Aivaliotika


## ***Add the Target Column***

In [9]:
# Check if a business exists for each (NACE, Neighborhood) pair
business_counts = (
    business_df.groupby(['NACE Code', 'Neighborhood'])
    .size()
    .reset_index(name='count')
)

In [10]:
# Merge to mark existing combinations
result_df = pd.merge(
    combinations_df,
    business_counts,
    on=['NACE Code', 'Neighborhood'],
    how='left'
)

In [11]:
# Assign target: 1 if count >= 1, else 0
result_df['target'] = (result_df['count'] >= 1).astype(int)

In [12]:
result_df = result_df.drop(columns=['count'])

In [13]:
print(result_df.shape)
result_df.head()

(14688, 3)


,NACE Code,Neighborhood,target
0,47.76,Dimini,0
1,47.76,Xrisi Akti Panagias,0
2,47.76,Sesklo,0
3,47.76,Agioi Anargiroi,0
4,47.76,Aivaliotika,1


In [14]:
print(f"Total pairs: {len(result_df)}")
print(f"Positive cases (1): {result_df['target'].sum()}")
print(f"Negative cases (0): {len(result_df) - result_df['target'].sum()}")

Total pairs: 14688
Positive cases (1): 1702
Negative cases (0): 12986


## ***Add the Neighborhood Features***

### ***Merge the Data***

In [18]:
# Merge on Neighborhood
merged_df = result_df.merge(neighborhood_df, on="Neighborhood", how="left")

In [19]:
merged_df = merged_df.drop(columns=['Geometry', 'Neighborhood_Greek'])

In [20]:
print(merged_df.shape)
print(merged_df.columns)
merged_df.head()

(14688, 12)
Index(['NACE Code', 'Neighborhood', 'target', 'Municipal Community',
       'Centroid_x', 'Centroid_y', 'Neighborhood_Area_km2',
       'distance_to_volos_center_km', 'distance_to_volos_port_km',
       'dist_to_main_road_km', 'dist_to_bus_stop_km', 'dist_to_university_km'],
      dtype='object')


,NACE Code,Neighborhood,target,Municipal Community,Centroid_x,Centroid_y,Neighborhood_Area_km2,distance_to_volos_center_km,distance_to_volos_port_km,dist_to_main_road_km,dist_to_bus_stop_km,dist_to_university_km
0,47.76,Dimini,0,Δημοτική Κοινότητα Διμηνίου,22.882945,39.349112,37.344776,6.154015,5.418577,0.443434,1.569912,4.273031
1,47.76,Xrisi Akti Panagias,0,Δημοτική Κοινότητα Σέσκλου,22.836899,39.305430,9.983032,11.932475,11.015584,2.299921,2.410375,10.100771
2,47.76,Sesklo,0,Δημοτική Κοινότητα Σέσκλου,22.838168,39.353051,27.367312,9.814666,9.191001,0.482308,4.965561,7.989610
3,47.76,Agioi Anargiroi,0,Δημοτική Κοινότητα Βόλου,22.924059,39.366937,0.774436,2.296184,1.978856,0.058142,0.063879,0.866021
4,47.76,Aivaliotika,1,Δημοτική Κοινότητα Βόλου,22.922660,39.343696,4.848904,3.508166,2.529372,0.224486,0.246542,1.913456


### ***Save the Data***

In [22]:
# Save to CSV
merged_df.to_csv("C:\\Users\\Giorgos\\Desktop\\HMMY\\10ο Εξάμηνο\\Διπλωματική\\6. Ground Truths\\1. Supervised Approach\\1. Approach 1\\Extracted CSV Files\\ML_model_dataset.csv", index=False)

## ***Add the Municipal Community Features***

### ***Merge the Data***

In [23]:
# Perform the merge
merged_df = merged_df.merge(
    municipal_community_df,
    left_on='Municipal Community',
    right_on='ΔΗΜΟΤΙΚΗ ΚΟΙΝΟΤΗΤΑ',
    how='left' 
)

In [ ]:
merged_df = merged_df.drop(columns=['ΔΗΜΟΤΙΚΗ ΚΟΙΝΟΤΗΤΑ'])

In [26]:
merged_df = merged_df.drop(columns=['ΔΗΜΟΤΙΚΗ ΚΟΙΝΟΤΗΤΑ ΕΛΛ'])

In [27]:
print(merged_df.shape)
print(merged_df.columns)
merged_df.head()

(14688, 32)
Index(['NACE Code', 'Neighborhood', 'target', 'Municipal Community',
       'Centroid_x', 'Centroid_y', 'Neighborhood_Area_km2',
       'distance_to_volos_center_km', 'distance_to_volos_port_km',
       'dist_to_main_road_km', 'dist_to_bus_stop_km', 'dist_to_university_km',
       'ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ', 'ΔΗΜΟΣ', 'ΔΗΜΟΤΙΚΗ ΕΝΟΤΗΤΑ', 'Population',
       'Άγαμοι_pct', 'Έγγαμοι_pct', 'Χήροι_pct', 'Διαζευγμένοι_pct',
       'age_0_14_pct', 'age_15_64_pct', 'age_65_plus_pct', 'low_education_pct',
       'medium_education_pct', 'high_education_pct', 'unemployment_rate',
       'labor_force_participation_rate', 'primary_sector_pct',
       'secondary_sector_pct', 'tertiary_sector_pct', 'Area_km2'],
      dtype='object')


,NACE Code,Neighborhood,target,Municipal Community,Centroid_x,Centroid_y,Neighborhood_Area_km2,distance_to_volos_center_km,distance_to_volos_port_km,dist_to_main_road_km,...,age_65_plus_pct,low_education_pct,medium_education_pct,high_education_pct,unemployment_rate,labor_force_participation_rate,primary_sector_pct,secondary_sector_pct,tertiary_sector_pct,Area_km2
0,47.76,Dimini,0,Δημοτική Κοινότητα Διμηνίου,22.882945,39.349112,37.344776,6.154015,5.418577,0.443434,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,47.76,Xrisi Akti Panagias,0,Δημοτική Κοινότητα Σέσκλου,22.836899,39.305430,9.983032,11.932475,11.015584,2.299921,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,47.76,Sesklo,0,Δημοτική Κοινότητα Σέσκλου,22.838168,39.353051,27.367312,9.814666,9.191001,0.482308,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,47.76,Agioi Anargiroi,0,Δημοτική Κοινότητα Βόλου,22.924059,39.366937,0.774436,2.296184,1.978856,0.058142,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,47.76,Aivaliotika,1,Δημοτική Κοινότητα Βόλου,22.922660,39.343696,4.848904,3.508166,2.529372,0.224486,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### ***Save the Data***

In [30]:
# Save to CSV
merged_df.to_csv("C:\\Users\\Giorgos\\Desktop\\HMMY\\10ο Εξάμηνο\\Διπλωματική\\6. Ground Truths\\1. Supervised Approach\\2. Approach 2 (won)\\Extracted CSV Files\\ML_model_dataset.csv", index=False)